# Building a ReAct Agent — The Foundational Agentic Pattern

## What This Notebook Teaches You

This is the **most important notebook** in the course. Everything from Notebooks 02–04 converges here: the agent loop, tool use, and structured output parsing combine into **ReAct** — the pattern that powers modern AI agents.

**ReAct** (Reasoning + Acting) was introduced by Yao et al. (2022). The key insight: instead of having the LLM think *then* act, or act *then* think, ReAct **interleaves** them. Each step produces a **Thought** (reasoning), an **Action** (tool call), and an **Observation** (tool result), creating a trace that is both interpretable and effective.

By the end of this notebook, you will be able to:

1. **Explain the ReAct paradigm** — why interleaving reasoning and action improves performance
2. **Build infrastructure** — compact, self-contained ToolRegistry and OutputParser
3. **Define a knowledge base** — 30 facts across science, history, and geography
4. **Implement 8 tools** — from calculation to knowledge search to fact checking
5. **Build a complete ReActAgent** — the full class with run, step, parse, execute, and trace
6. **Solve diverse tasks** — from simple lookups to complex multi-tool synthesis
7. **Compare ReAct vs alternatives** — single prompt, CoT, and ReAct on the same tasks
8. **Diagnose failures** — understand when and why ReAct agents fail

---

> **Prerequisites**: Complete Notebooks 02–04 (Agent Loop, Tool Use, Structured Output).
> **Runtime**: Google Colab with T4 GPU (free tier).
> **Time**: ~60–90 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. The ReAct Paradigm

### 1.1 — The Research

**"ReAct: Synergizing Reasoning and Acting in Language Models"** (Yao et al., 2022) showed that interleaving reasoning with actions outperforms either alone:

| Approach | How it works | Limitation |
|----------|-------------|------------|
| **Reasoning only** (CoT) | Think step-by-step, then answer | No access to external information |
| **Acting only** (tool use) | Call tools directly | No planning or reflection |
| **ReAct** | Think → Act → Observe → Repeat | Best of both: plans, then verifies |

### 1.2 — The Thought–Action–Observation Loop

```
┌───────────────────────────────────────────────────────────┐
│                    THE ReAct LOOP                         │
│                                                           │
│   Task: "What is the population of the capital of France?"│
│                                                           │
│   ┌─────────────────────────────────────────────┐         │
│   │ Step 1                                      │         │
│   │  Thought: I need to find the capital of     │         │
│   │          France first, then look up its     │         │
│   │          population.                        │         │
│   │  Action:  search_knowledge("capital France")│         │
│   │  Observation: "Paris is the capital of      │         │
│   │              France"                        │         │
│   └──────────────────────┬──────────────────────┘         │
│                          ▼                                │
│   ┌─────────────────────────────────────────────┐         │
│   │ Step 2                                      │         │
│   │  Thought: The capital is Paris. Now I need  │         │
│   │          to find its population.            │         │
│   │  Action:  search_knowledge("population      │         │
│   │           Paris")                           │         │
│   │  Observation: "Paris has a population of    │         │
│   │              approximately 2.1 million"     │         │
│   └──────────────────────┬──────────────────────┘         │
│                          ▼                                │
│   ┌─────────────────────────────────────────────┐         │
│   │ Step 3                                      │         │
│   │  Thought: I have both pieces. The capital   │         │
│   │          of France is Paris with ~2.1M      │         │
│   │          population.                        │         │
│   │  Action:  finish("The capital of France is  │         │
│   │          Paris, with ~2.1 million people.") │         │
│   └─────────────────────────────────────────────┘         │
└───────────────────────────────────────────────────────────┘
```

### 1.3 — Why Interleaving Works

1. **Thoughts guide actions**: The LLM plans *what to do next* before doing it
2. **Observations ground reasoning**: Real tool results prevent hallucination
3. **Traces are interpretable**: Every decision is visible and debuggable
4. **Error recovery**: The LLM can recognize when a tool result is unexpected and adjust

## 2. Infrastructure — Self-Contained Tool & Parser Components

We build compact, self-contained versions of the ToolRegistry and OutputParser from previous notebooks.

In [ ]:
# ===== Compact ToolRegistry =====
class ToolRegistry:
    """Minimal tool registry with auto-schema extraction."""

    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def register(self, func: Callable = None, *, name: str = None,
                 description: str = None) -> Callable:
        """Register a function as a tool (decorator-compatible)."""
        def decorator(f):
            tool_name = name or f.__name__
            tool_desc = description or (f.__doc__ or "").strip().split("\n")[0]
            sig = inspect.signature(f)
            hints = get_type_hints(f) if hasattr(f, '__annotations__') else {}
            type_map = {str: "string", int: "int", float: "float", bool: "bool", list: "list"}

            params = {}
            for pname, param in sig.parameters.items():
                ptype = hints.get(pname, str)
                params[pname] = {
                    "type": type_map.get(ptype, "string"),
                    "required": param.default is inspect.Parameter.empty,
                    "default": None if param.default is inspect.Parameter.empty else param.default,
                }

            self.tools[tool_name] = {
                "name": tool_name,
                "description": tool_desc,
                "parameters": params,
                "function": f,
            }
            return f

        if func is not None:
            return decorator(func)
        return decorator

    def call(self, name: str, arguments: Dict[str, Any]) -> str:
        """Call a registered tool by name."""
        if name not in self.tools:
            available = ", ".join(self.tools.keys())
            return f"Error: Unknown tool '{name}'. Available: {available}"
        tool = self.tools[name]
        try:
            sig = inspect.signature(tool["function"])
            valid_args = {k: v for k, v in arguments.items() if k in sig.parameters}
            result = tool["function"](**valid_args)
            return str(result)
        except Exception as e:
            return f"Error executing {name}: {str(e)}"

    def format_for_prompt(self) -> str:
        """Format all tools for the system prompt."""
        lines = []
        for t in self.tools.values():
            params = ", ".join(
                f"{p}: {info['type']}" + ("" if info["required"] else f"={info['default']}")
                for p, info in t["parameters"].items()
            )
            lines.append(f"  - {t['name']}({params}): {t['description']}")
        return "\n".join(lines)

print("✓ ToolRegistry defined")

In [ ]:
# ===== Compact OutputParser =====
class OutputParser:
    """Robust JSON parser with multiple fallback strategies."""

    @staticmethod
    def parse(text: str) -> Optional[Dict[str, Any]]:
        """Extract a JSON object from LLM output."""
        text = text.strip()

        # Strategy 1: Direct parse
        try:
            result = json.loads(text)
            if isinstance(result, dict):
                return result
        except (json.JSONDecodeError, TypeError):
            pass

        # Strategy 2: Code block extraction
        code_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', text)
        if code_match:
            try:
                result = json.loads(code_match.group(1).strip())
                if isinstance(result, dict):
                    return result
            except (json.JSONDecodeError, TypeError):
                pass

        # Strategy 3: Brace matching with nesting
        depth = 0
        start = None
        for i, ch in enumerate(text):
            if ch == '{':
                if depth == 0:
                    start = i
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0 and start is not None:
                    try:
                        result = json.loads(text[start:i+1])
                        if isinstance(result, dict) and "action" in result:
                            return result
                    except (json.JSONDecodeError, TypeError):
                        pass
                    start = None

        # Strategy 4: JSON repair (single quotes, trailing commas)
        for candidate in re.finditer(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text):
            repaired = candidate.group()
            repaired = re.sub(r"'([^']*)'", r'"\1"', repaired)
            repaired = re.sub(r',\s*([}\]])', r'\1', repaired)
            try:
                result = json.loads(repaired)
                if isinstance(result, dict):
                    return result
            except (json.JSONDecodeError, TypeError):
                pass

        return None

# Test
tests = [
    '{"action": "think", "thought": "test"}',
    'Response: {"action": "answer", "answer": "42"} done.',
    "{'action': 'think', 'thought': 'single quotes'}",
]
for t in tests:
    print(f"  {t[:50]:50s} → {OutputParser.parse(t)}")
print("\n✓ OutputParser defined")

## 3. The Knowledge Base — 30 Facts

Our agent needs a world to reason about. We create a knowledge base of 30 facts spanning science, history, and geography.

In [ ]:
KNOWLEDGE_BASE = {
    # Science (10 facts)
    "speed of light": "The speed of light in vacuum is approximately 299,792,458 meters per second (about 3 × 10^8 m/s).",
    "photosynthesis": "Photosynthesis is the process by which plants convert carbon dioxide and water into glucose and oxygen using sunlight.",
    "DNA": "DNA (deoxyribonucleic acid) is a double-helix molecule that carries genetic instructions for the development and functioning of living organisms.",
    "gravity": "Gravity is a fundamental force of attraction between objects with mass. On Earth, gravitational acceleration is approximately 9.8 m/s².",
    "water boiling point": "Water boils at 100°C (212°F) at standard atmospheric pressure (1 atm). The boiling point decreases at higher altitudes.",
    "human bones": "An adult human body has 206 bones. Babies are born with approximately 270 bones, which fuse together as they grow.",
    "periodic table": "The periodic table has 118 confirmed elements. Hydrogen (H) is the first element, and Oganesson (Og) is the 118th.",
    "mitochondria": "Mitochondria are organelles found in eukaryotic cells that generate most of the cell's supply of ATP, the energy currency.",
    "evolution": "Evolution by natural selection, proposed by Charles Darwin, explains how species change over time through inherited variations that affect survival.",
    "atoms": "An atom consists of a nucleus (protons and neutrons) surrounded by electrons. Protons carry positive charge, electrons carry negative charge.",

    # History (10 facts)
    "moon landing": "The first moon landing was on July 20, 1969. Apollo 11 astronauts Neil Armstrong and Buzz Aldrin walked on the lunar surface.",
    "world war 2": "World War 2 lasted from 1939 to 1945. It involved most of the world's nations and resulted in an estimated 70-85 million fatalities.",
    "printing press": "Johannes Gutenberg invented the movable-type printing press around 1440, revolutionizing the spread of information in Europe.",
    "french revolution": "The French Revolution began in 1789 and ended in 1799. It led to the end of the monarchy and the rise of Napoleon Bonaparte.",
    "ancient egypt": "Ancient Egyptian civilization lasted from about 3100 BC to 30 BC. The Great Pyramid of Giza was built around 2560 BC.",
    "roman empire": "The Roman Empire at its peak (around 117 AD under Emperor Trajan) controlled about 5 million square kilometers.",
    "industrial revolution": "The Industrial Revolution began in Britain in the late 18th century (around 1760) and transformed manufacturing, transportation, and society.",
    "cold war": "The Cold War (1947-1991) was a geopolitical tension between the United States and the Soviet Union. It ended with the dissolution of the USSR.",
    "internet": "The internet originated from ARPANET in 1969. The World Wide Web was invented by Tim Berners-Lee in 1989.",
    "renaissance": "The Renaissance was a cultural movement from the 14th to 17th century, originating in Italy, marked by renewed interest in classical art and learning.",

    # Geography (10 facts)
    "mount everest": "Mount Everest is the tallest mountain on Earth at 8,849 meters (29,032 feet) above sea level, located in the Himalayas on the Nepal-Tibet border.",
    "amazon river": "The Amazon River is the largest river by discharge volume and the second longest at approximately 6,400 km. It flows through South America.",
    "sahara desert": "The Sahara is the largest hot desert in the world, covering about 9.2 million square kilometers in North Africa.",
    "pacific ocean": "The Pacific Ocean is the largest and deepest ocean, covering about 165.25 million square kilometers — more than all land area combined.",
    "capital of france": "Paris is the capital of France, with a population of approximately 2.1 million in the city proper and 12 million in the metropolitan area.",
    "great wall of china": "The Great Wall of China is approximately 21,196 kilometers long. Construction began in the 7th century BC and continued for centuries.",
    "australia": "Australia is both a country and a continent. It has an area of about 7.7 million square kilometers and a population of approximately 26 million.",
    "nile river": "The Nile River is the longest river in the world at approximately 6,650 km, flowing through northeastern Africa.",
    "antarctica": "Antarctica is the coldest, driest, and windiest continent. It has no permanent residents and is covered by ice averaging 2.16 km thick.",
    "mariana trench": "The Mariana Trench is the deepest part of the ocean, reaching approximately 10,994 meters (36,070 feet) at Challenger Deep.",
}

print(f"Knowledge base loaded: {len(KNOWLEDGE_BASE)} facts")
print(f"\nCategories:")
print(f"  Science:   {', '.join(list(KNOWLEDGE_BASE.keys())[:5])}...")
print(f"  History:   {', '.join(list(KNOWLEDGE_BASE.keys())[10:15])}...")
print(f"  Geography: {', '.join(list(KNOWLEDGE_BASE.keys())[20:25])}...")

## 4. Building 8 Tools for the ReAct Agent

Each tool serves a different purpose, giving the agent diverse capabilities.

In [ ]:
registry = ToolRegistry()

@registry.register(description="Evaluate a mathematical expression. Supports +, -, *, /, **, parentheses. Example: '(10 + 5) * 3'")
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    allowed = set("0123456789+-*/.() ,eE")
    if not all(c in allowed for c in expression):
        return f"Error: Expression contains invalid characters. Use only numbers and +, -, *, /, **, ()."
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@registry.register(description="Search the knowledge base for facts about science, history, and geography. Use a short keyword query like 'moon landing' or 'photosynthesis'.")
def search_knowledge(query: str) -> str:
    """Search the knowledge base for relevant facts."""
    query_lower = query.lower().strip()

    # Exact match
    if query_lower in KNOWLEDGE_BASE:
        return KNOWLEDGE_BASE[query_lower]

    # Substring match: find keys that contain query or vice versa
    matches = []
    for key, fact in KNOWLEDGE_BASE.items():
        if query_lower in key or key in query_lower:
            matches.append((key, fact))
        elif any(word in key for word in query_lower.split() if len(word) > 3):
            matches.append((key, fact))

    if matches:
        results = []
        for key, fact in matches[:3]:
            results.append(f"[{key}] {fact}")
        return "\n".join(results)

    # No match — search in fact text
    text_matches = []
    for key, fact in KNOWLEDGE_BASE.items():
        if any(word in fact.lower() for word in query_lower.split() if len(word) > 3):
            text_matches.append((key, fact))

    if text_matches:
        results = []
        for key, fact in text_matches[:2]:
            results.append(f"[{key}] {fact}")
        return "\n".join(results)

    return f"No results found for '{query}'. Try different keywords."

@registry.register(description="Get the current date and time in YYYY-MM-DD HH:MM:SS format.")
def get_date() -> str:
    """Return current date and time."""
    from datetime import datetime
    now = datetime.now()
    return f"Current date and time: {now.strftime('%Y-%m-%d %H:%M:%S')} ({now.strftime('%A')})"

@registry.register(description="Analyze a text string. Returns character count, word count, sentence count, unique words, and average word length.")
def string_analyzer(text: str) -> str:
    """Analyze text and return statistics."""
    chars = len(text)
    words = text.split()
    word_count = len(words)
    sentences = max(len(re.split(r'[.!?]+', text.strip())) - 1, 1)
    unique = len(set(w.lower().strip(".,!?;:'\"") for w in words))
    avg_len = sum(len(w.strip(".,!?;:'\"")) for w in words) / max(word_count, 1)
    return (f"Characters: {chars}, Words: {word_count}, Sentences: {sentences}, "
            f"Unique words: {unique}, Avg word length: {avg_len:.1f}")

@registry.register(description="Perform operations on a comma-separated list: sort, reverse, count, unique, min, max. Example: operation='sort', items='3,1,2'")
def list_operations(operation: str, items: str) -> str:
    """Perform operations on comma-separated lists."""
    parts = [x.strip() for x in items.split(",")]
    op = operation.lower().strip()
    try:
        if op == "sort":
            try:
                return "Sorted: " + ", ".join(sorted(parts, key=float))
            except ValueError:
                return "Sorted: " + ", ".join(sorted(parts))
        elif op == "reverse":
            return "Reversed: " + ", ".join(reversed(parts))
        elif op == "count":
            return f"Count: {len(parts)}"
        elif op == "unique":
            return "Unique: " + ", ".join(dict.fromkeys(parts))
        elif op == "min":
            try:
                return f"Min: {min(parts, key=float)}"
            except ValueError:
                return f"Min (alphabetic): {min(parts)}"
        elif op == "max":
            try:
                return f"Max: {max(parts, key=float)}"
            except ValueError:
                return f"Max (alphabetic): {max(parts)}"
        else:
            return f"Unknown operation '{op}'. Use: sort, reverse, count, unique, min, max."
    except Exception as e:
        return f"Error: {str(e)}"

@registry.register(description="Compare two numbers. Returns which is larger, the difference, and the ratio. Example: a='10', b='3'")
def compare_numbers(a: str, b: str) -> str:
    """Compare two numbers."""
    try:
        x, y = float(a), float(b)
        diff = abs(x - y)
        ratio = x / y if y != 0 else float('inf')
        if x > y:
            comparison = f"{x} is greater than {y}"
        elif y > x:
            comparison = f"{y} is greater than {x}"
        else:
            comparison = f"{x} and {y} are equal"
        return f"{comparison}. Difference: {diff}. Ratio: {ratio:.4f}"
    except ValueError:
        return f"Error: Could not parse '{a}' and '{b}' as numbers."

@registry.register(description="Transform text: uppercase, lowercase, title_case, reverse, word_reverse, or count_chars. Example: operation='uppercase', text='hello'")
def text_transformer(operation: str, text: str) -> str:
    """Transform text in various ways."""
    op = operation.lower().strip()
    if op == "uppercase":
        return text.upper()
    elif op == "lowercase":
        return text.lower()
    elif op == "title_case":
        return text.title()
    elif op == "reverse":
        return text[::-1]
    elif op == "word_reverse":
        return " ".join(text.split()[::-1])
    elif op == "count_chars":
        from collections import Counter
        counts = Counter(text.lower())
        top = sorted(counts.items(), key=lambda x: -x[1])[:10]
        return "Character frequencies: " + ", ".join(f"'{c}': {n}" for c, n in top)
    else:
        return f"Unknown operation '{op}'. Use: uppercase, lowercase, title_case, reverse, word_reverse, count_chars."

@registry.register(description="Check a fact claim against the knowledge base. Returns whether the claim is supported, contradicted, or not found.")
def fact_checker(claim: str) -> str:
    """Check a factual claim against the knowledge base."""
    claim_lower = claim.lower()

    # Search knowledge base for related facts
    related = []
    for key, fact in KNOWLEDGE_BASE.items():
        # Check keyword overlap
        key_words = set(key.split())
        claim_words = set(claim_lower.split())
        overlap = key_words & claim_words
        if overlap or any(w in fact.lower() for w in claim_words if len(w) > 4):
            related.append((key, fact))

    if not related:
        return f"No relevant facts found in the knowledge base to verify: '{claim}'"

    facts_text = "\n".join(f"  [{k}] {f}" for k, f in related[:3])
    return f"Related facts found:\n{facts_text}\nCompare these facts with the claim to assess accuracy."

# Show all tools
print(f"Registered {len(registry.tools)} tools:\n")
print(registry.format_for_prompt())

## 5. The ReAct System Prompt

The system prompt is critical — it defines the agent's behavior, output format, and tool usage protocol.

In [ ]:
def build_react_prompt(registry: ToolRegistry) -> str:
    """Build the ReAct system prompt with tool descriptions."""
    tools_desc = registry.format_for_prompt()
    return f"""You are a ReAct agent that solves tasks by interleaving Thought, Action, and Observation steps.

Available tools:
{tools_desc}

On EACH turn, respond with EXACTLY one JSON object. Choose one of these formats:

1. To think/reason about the problem:
{{"action": "think", "thought": "your detailed reasoning about what to do next"}}

2. To use a tool:
{{"action": "tool", "tool": "<tool_name>", "arguments": {{"param1": "value1"}}}}

3. To give the final answer (ONLY when you have all needed information):
{{"action": "answer", "answer": "your complete final answer"}}

RULES:
- ALWAYS think before using a tool (plan what you need).
- ALWAYS use tools for facts, calculations, and data — do NOT guess or make things up.
- After receiving a tool result (Observation), reason about it before deciding the next action.
- When you have enough information, provide a clear final answer.
- Respond with ONLY the JSON object, no other text.
- Be systematic: break complex tasks into steps.

Example trace:
  Turn 1: {{"action": "think", "thought": "I need to find X, then calculate Y."}}
  Turn 2: {{"action": "tool", "tool": "search_knowledge", "arguments": {{"query": "X"}}}}
  [Observation: X is ...]
  Turn 3: {{"action": "think", "thought": "Now I know X. Let me calculate Y."}}
  Turn 4: {{"action": "tool", "tool": "calculator", "arguments": {{"expression": "..."}}}}
  [Observation: Result: ...]
  Turn 5: {{"action": "answer", "answer": "Based on my research, ..."}}"""

react_prompt = build_react_prompt(registry)
print(f"ReAct system prompt ({len(react_prompt)} chars):")
print(react_prompt[:600])
print("...")

## 6. Building the ReActAgent Class

This is the complete implementation — the culmination of everything we've built.

In [ ]:
class ReActAgent:
    """Complete ReAct agent: Thought → Action → Observation loop."""

    def __init__(self, registry: ToolRegistry, max_steps: int = 10,
                 temperature: float = 0.5, verbose: bool = True):
        self.registry = registry
        self.max_steps = max_steps
        self.temperature = temperature
        self.verbose = verbose
        self.system_prompt = build_react_prompt(registry)

    def run(self, task: str) -> Dict[str, Any]:
        """Execute the ReAct loop on a task."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Task: {task}"},
        ]

        trace = []
        final_answer = None
        start_time = time.time()

        if self.verbose:
            print(f"\n{'═'*70}")
            print(f"  ReAct Agent — {task}")
            print(f"{'═'*70}")

        for step_num in range(1, self.max_steps + 1):
            step_start = time.time()

            # Generate LLM response
            response = generate(messages, max_new_tokens=512, temperature=self.temperature)
            parsed = self._parse_response(response)

            action = parsed.get("action", "unknown")
            step_time = time.time() - step_start

            step_record = {
                "step": step_num,
                "action": action,
                "parsed": parsed,
                "raw": response[:500],
                "time": step_time,
                "observation": None,
            }

            if action == "think":
                thought = parsed.get("thought", "")
                if self.verbose:
                    print(f"\n  💭 Step {step_num} — THINK ({step_time:.1f}s)")
                    self._print_wrapped(thought, indent=5)
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": "Continue with your next action."})

            elif action == "tool":
                tool_name = parsed.get("tool", "unknown")
                arguments = parsed.get("arguments", {})
                if not isinstance(arguments, dict):
                    arguments = {}

                # Execute tool
                observation = self._execute_tool(tool_name, arguments)
                step_record["observation"] = observation

                if self.verbose:
                    print(f"\n  🔧 Step {step_num} — TOOL: {tool_name} ({step_time:.1f}s)")
                    print(f"       Args: {arguments}")
                    print(f"       Obs:  {observation[:150]}{'...' if len(observation) > 150 else ''}")

                messages.append({"role": "assistant", "content": response})
                messages.append({
                    "role": "user",
                    "content": f"Observation: {observation}\n\nReason about this result and decide your next action."
                })

            elif action == "answer":
                final_answer = parsed.get("answer", "")
                if self.verbose:
                    print(f"\n  ✅ Step {step_num} — ANSWER ({step_time:.1f}s)")
                    self._print_wrapped(final_answer, indent=5)
                step_record["observation"] = "DONE"
                trace.append(step_record)
                break

            else:
                if self.verbose:
                    print(f"\n  ❓ Step {step_num} — UNKNOWN ({step_time:.1f}s): {response[:100]}")
                messages.append({"role": "assistant", "content": response})
                messages.append({
                    "role": "user",
                    "content": "Please respond with a valid JSON object with 'action' field."
                })

            trace.append(step_record)

        elapsed = time.time() - start_time

        if final_answer is None:
            final_answer = self._extract_best_answer(trace)

        result = {
            "task": task,
            "answer": final_answer,
            "steps": len(trace),
            "trace": trace,
            "tool_calls": [s for s in trace if s["action"] == "tool"],
            "think_steps": [s for s in trace if s["action"] == "think"],
            "elapsed": elapsed,
        }

        if self.verbose:
            print(f"\n{'─'*70}")
            print(f"  Completed: {len(trace)} steps, "
                  f"{len(result['tool_calls'])} tool calls, {elapsed:.1f}s")
            print(f"{'═'*70}")

        return result

    def _parse_response(self, text: str) -> Dict[str, Any]:
        """Parse LLM response with OutputParser."""
        result = OutputParser.parse(text)
        if result and "action" in result:
            return result
        # Fallback: treat as thought
        return {"action": "think", "thought": text[:500]}

    def _execute_tool(self, name: str, arguments: Dict[str, Any]) -> str:
        """Execute a tool and return the observation."""
        return self.registry.call(name, arguments)

    def _extract_best_answer(self, trace: List[Dict]) -> str:
        """Extract best answer from trace when agent didn't explicitly answer."""
        for step in reversed(trace):
            if step["action"] == "answer":
                return step["parsed"].get("answer", "")
        for step in reversed(trace):
            if step.get("observation") and step["observation"] != "DONE":
                return f"[Incomplete] Last observation: {step['observation'][:200]}"
        for step in reversed(trace):
            if step["action"] == "think":
                return f"[Incomplete] Last thought: {step['parsed'].get('thought', '')[:200]}"
        return "[No answer produced]"

    def _print_wrapped(self, text: str, indent: int = 5, width: int = 70):
        """Print text with word wrapping."""
        prefix = " " * indent
        words = text.split()
        line = prefix
        for word in words:
            if len(line) + len(word) + 1 > width + indent:
                print(line)
                line = prefix + word
            else:
                line = line + " " + word if line.strip() else prefix + word
        if line.strip():
            print(line)

    def get_trace(self) -> str:
        """Return a formatted trace string (call after run)."""
        return "Use the result dict's 'trace' key for step-by-step data."

print("✓ ReActAgent class defined")

## 7. Running the ReAct Agent on Diverse Tasks

We test on 8 tasks of increasing complexity to demonstrate the full range of ReAct behavior.

In [ ]:
agent = ReActAgent(registry, max_steps=8)

# Task 1: Simple fact lookup
print("\n" + "▶"*25 + " TASK 1: Simple Fact " + "◀"*25)
r1 = agent.run("What is the speed of light?")

In [ ]:
# Task 2: Calculation
print("\n" + "▶"*25 + " TASK 2: Calculation " + "◀"*25)
r2 = agent.run("What is 347 * 829 + 156?")

In [ ]:
# Task 3: Multi-step lookup
print("\n" + "▶"*25 + " TASK 3: Multi-Step " + "◀"*25)
r3 = agent.run(
    "What is the tallest mountain on Earth and how does its height compare "
    "to the depth of the deepest ocean point? Which is greater and by how much?"
)

In [ ]:
# Task 4: Text analysis
print("\n" + "▶"*25 + " TASK 4: Text Analysis " + "◀"*25)
r4 = agent.run(
    "Analyze this sentence and give me detailed statistics: "
    "'The quick brown fox jumps over the lazy dog near the sparkling river'"
)

In [ ]:
# Task 5: Fact verification
print("\n" + "▶"*25 + " TASK 5: Fact Verification " + "◀"*25)
r5 = agent.run(
    "Is it true that the Great Wall of China is visible from space? "
    "Check this claim against known facts."
)

In [ ]:
# Task 6: Multi-tool combination
print("\n" + "▶"*25 + " TASK 6: Multi-Tool " + "◀"*25)
r6 = agent.run(
    "Find out when World War 2 ended, calculate how many years ago that was "
    "(assume the current year is 2025), and tell me a fact about what was "
    "invented around that same era."
)

In [ ]:
# Task 7: Reasoning + tools
print("\n" + "▶"*25 + " TASK 7: Reasoning + Tools " + "◀"*25)
r7 = agent.run(
    "The Pacific Ocean is the largest ocean. How many times larger is it than "
    "the Sahara Desert? Search for both areas and calculate the ratio."
)

In [ ]:
# Task 8: Complex synthesis
print("\n" + "▶"*25 + " TASK 8: Complex Synthesis " + "◀"*25)
r8 = agent.run(
    "Compare the Nile River and the Amazon River: which is longer, which has "
    "more discharge volume? Sort these two rivers plus the length of the Great "
    "Wall of China by their numeric values from smallest to largest."
)

## 8. Grand Comparison — Single Prompt vs. CoT vs. ReAct

The ultimate test: the same 4 tasks solved three ways.

In [ ]:
comparison_tasks = [
    "What is the depth of the Mariana Trench in feet?",
    "How many years passed between the invention of the printing press and the first moon landing?",
    "Which is longer — the Nile River or the Amazon River — and by approximately how many kilometers?",
    "How many bones does an adult human have, and what is that number squared?",
]

def single_prompt(task):
    """Solve with a single prompt (no agent loop)."""
    messages = [
        {"role": "system", "content": "Answer the question concisely."},
        {"role": "user", "content": task},
    ]
    start = time.time()
    answer = generate(messages, max_new_tokens=256, temperature=0.3)
    return {"answer": answer.strip(), "time": time.time() - start, "steps": 1}

def chain_of_thought(task):
    """Solve with Chain of Thought (reasoning, no tools)."""
    messages = [
        {"role": "system", "content": "Think step by step, then give your final answer on the last line."},
        {"role": "user", "content": task},
    ]
    start = time.time()
    answer = generate(messages, max_new_tokens=512, temperature=0.3)
    return {"answer": answer.strip(), "time": time.time() - start, "steps": 1}

print("Grand Comparison: Single Prompt vs CoT vs ReAct")
print("=" * 70)

comparison_results = []
quiet_agent = ReActAgent(registry, max_steps=8, verbose=False)

for i, task in enumerate(comparison_tasks, 1):
    print(f"\nTask {i}: {task}")
    print("-" * 70)

    sp = single_prompt(task)
    cot = chain_of_thought(task)
    react = quiet_agent.run(task)

    print(f"  Single: {sp['answer'][:100]}")
    print(f"  CoT:    {cot['answer'][:100]}")
    print(f"  ReAct:  {react['answer'][:100]}")

    comparison_results.append({
        "task": task[:60],
        "single_time": sp["time"],
        "cot_time": cot["time"],
        "react_time": react["elapsed"],
        "react_steps": react["steps"],
        "react_tools": len(react["tool_calls"]),
    })

In [ ]:
# Summary table
print("\nComparison Summary")
print("=" * 85)
print(f"{'Task':<40} {'Single':<10} {'CoT':<10} {'ReAct':<10} {'Steps':<8} {'Tools':<6}")
print("-" * 85)
for r in comparison_results:
    print(f"{r['task'][:38]:<40} {r['single_time']:<10.1f} {r['cot_time']:<10.1f} "
          f"{r['react_time']:<10.1f} {r['react_steps']:<8} {r['react_tools']:<6}")

print(f"\nAvg time: Single={sum(r['single_time'] for r in comparison_results)/len(comparison_results):.1f}s, "
      f"CoT={sum(r['cot_time'] for r in comparison_results)/len(comparison_results):.1f}s, "
      f"ReAct={sum(r['react_time'] for r in comparison_results)/len(comparison_results):.1f}s")
print(f"ReAct avg steps: {sum(r['react_steps'] for r in comparison_results)/len(comparison_results):.1f}")
print(f"ReAct avg tools: {sum(r['react_tools'] for r in comparison_results)/len(comparison_results):.1f}")

## 9. Failure Analysis — When ReAct Struggles

No agent is perfect. Let's examine tasks that cause failures and understand why.

In [ ]:
failure_tasks = [
    # Ambiguous query — tool may not find relevant info
    "What is the average temperature on Mars compared to Earth?",
    # Requires multi-step calculation with intermediate reasoning
    "If the Nile is about 6650 km and I could walk 5 km per hour for 8 hours per day, how many days would it take to walk its length?",
    # Requires information not in our knowledge base
    "Who won the 2024 Nobel Prize in Physics?",
]

print("Failure Analysis — Tricky Tasks")
print("=" * 70)

failure_agent = ReActAgent(registry, max_steps=6, verbose=True)

failure_results = []
for task in failure_tasks:
    print(f"\n{'▶'*20} TRICKY TASK {'◀'*20}")
    result = failure_agent.run(task)
    failure_results.append(result)

    # Analyze what happened
    actions = [s["action"] for s in result["trace"]]
    tools_used = [s["parsed"].get("tool", "?") for s in result["trace"] if s["action"] == "tool"]
    failed_tools = [s for s in result["trace"] if s.get("observation", "").startswith("Error")]

    print(f"\n  Analysis:")
    print(f"    Actions: {' → '.join(actions)}")
    print(f"    Tools used: {tools_used}")
    print(f"    Failed tools: {len(failed_tools)}")
    print(f"    Got answer: {'Yes' if not result['answer'].startswith('[') else 'No'}")

## 10. Optimization Tips for ReAct Agents

### 10.1 — Temperature

Lower temperature → more deterministic tool calls. Higher → more creative reasoning.

| Temperature | Best for |
|-------------|----------|
| 0.1–0.3 | Factual tasks, tool calls |
| 0.4–0.6 | General purpose (our default) |
| 0.7–1.0 | Creative tasks, brainstorming |

### 10.2 — Prompt Length

The system prompt is the largest token cost. Optimization strategies:
- **Compress tool descriptions**: Use concise descriptions, not paragraphs
- **Limit examples**: One example trace is often enough
- **Dynamic tool loading**: Only include tools relevant to the current task

### 10.3 — Token Budgeting

```
Total tokens per step ≈ system_prompt + conversation_history + new_generation

For our setup:
  System prompt:    ~500 tokens
  Per step history: ~100-200 tokens
  Generation:       ~100-200 tokens

  8-step run ≈ 500 + 8*(150+150) = ~2,900 tokens
  vs single prompt ≈ 500 + 200 = ~700 tokens
```

ReAct uses **4x more tokens** than a single prompt. The investment pays off when:
- Tasks require **external information** (tools)
- Tasks require **multi-step reasoning** (decomposition)
- **Accuracy matters** more than speed

In [ ]:
# Demonstrate temperature effect
print("Temperature Experiment")
print("=" * 60)
task = "How many bones does an adult human have?"

for temp in [0.1, 0.5, 0.9]:
    temp_agent = ReActAgent(registry, max_steps=6, temperature=temp, verbose=False)
    result = temp_agent.run(task)
    actions = [s["action"] for s in result["trace"]]
    print(f"\n  Temp={temp}: {result['steps']} steps, actions={' → '.join(actions)}")
    print(f"    Answer: {result['answer'][:100]}")

## 11. Aggregate Analysis of All Runs

In [ ]:
# Collect all results from the 8 main tasks
all_results = [r1, r2, r3, r4, r5, r6, r7, r8]

print("Aggregate Analysis — 8 ReAct Tasks")
print("=" * 60)

total_steps = sum(r["steps"] for r in all_results)
total_tools = sum(len(r["tool_calls"]) for r in all_results)
total_thinks = sum(len(r["think_steps"]) for r in all_results)
total_time = sum(r["elapsed"] for r in all_results)

print(f"\n  Total steps:      {total_steps}")
print(f"  Total tool calls: {total_tools}")
print(f"  Total think steps:{total_thinks}")
print(f"  Total time:       {total_time:.1f}s")
print(f"  Avg steps/task:   {total_steps/len(all_results):.1f}")
print(f"  Avg tools/task:   {total_tools/len(all_results):.1f}")
print(f"  Avg time/task:    {total_time/len(all_results):.1f}s")

# Tool usage frequency
tool_freq = defaultdict(int)
for r in all_results:
    for tc in r["tool_calls"]:
        tool_freq[tc["parsed"].get("tool", "?")] += 1

print(f"\n  Tool usage:")
for tool_name, count in sorted(tool_freq.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"    {tool_name:<20} {count:>3} {bar}")

# Steps distribution
print(f"\n  Steps per task:")
for r in all_results:
    bar = "█" * r["steps"]
    print(f"    {r['task'][:35]:<37} {r['steps']:>2} steps {bar}")

## 12. Key Takeaways

### What We Built
A complete **ReAct agent** with:
- **ToolRegistry** — 8 tools with auto-schema extraction
- **OutputParser** — 4-strategy JSON parser with fallbacks
- **Knowledge Base** — 30 facts across science, history, geography
- **ReActAgent** — full Thought–Action–Observation loop
- **Comparison framework** — single prompt vs CoT vs ReAct

### Core Insights

1. **ReAct = Reasoning + Acting.** Interleaving thoughts with tool use produces better results than either alone. Thoughts guide actions; observations ground reasoning.

2. **Tools prevent hallucination.** When the agent uses `calculator` instead of mental math, or `search_knowledge` instead of guessing, accuracy improves dramatically.

3. **The trace is the product.** Unlike a black-box model, a ReAct agent shows *every decision*. This makes debugging, auditing, and improvement possible.

4. **Token cost is the tradeoff.** ReAct uses 3–5x more tokens than a single prompt. This is justified when accuracy matters, tasks are complex, or external data is needed.

5. **Failure modes are predictable.** Missing knowledge, wrong tool selection, and parsing failures account for most errors. Each can be addressed with better prompts, more tools, or better parsing.

6. **ReAct is foundational.** Every advanced pattern — multi-agent, planning, reflection, memory — builds on this Thought–Action–Observation loop. Master it here.

### The Road Ahead

| Notebook | What it adds |
|----------|-------------|
| **06** | Memory — agents that learn from past experience |
| **07** | Planning — agents that decompose before executing |
| **08** | Multi-agent — teams of specialized agents |
| **09–12** | Advanced patterns: reflection, evaluation, orchestration, deployment |

---

```
Notebook 05 Complete — Building a ReAct Agent
The foundational pattern. Everything builds from here.
```